# 🇮🇳 01 — Sarathi AI: India Data Collection

Downloads all source data: IPC, Gita, Tourism Stats, Hotels, Wikipedia, Social (public only)

**How to use in Google Colab:**
1. Go to https://colab.research.google.com/
2. File → Upload notebook (upload this file)
3. Runtime → Change runtime type → GPU (T4 recommended)
4. Run all cells top to bottom with Shift+Enter

In [ ]:
!nvidia-smi || true
!pip install -q wikipedia pypdf requests datasets huggingface_hub

In [ ]:
import os, json, requests, time, wikipedia
from pathlib import Path

RAW = Path('data/raw')
RAW.mkdir(parents=True, exist_ok=True)
(RAW / 'social').mkdir(exist_ok=True)
print('Folders ready')

## Step 1 — Download Government & Legal PDFs

In [ ]:
def dl(url, dest, label):
    print(f'Downloading {label}...')
    r = requests.get(url, timeout=30, headers={'User-Agent':'SarathiAI/1.0'})
    open(dest,'wb').write(r.content)
    print(f'  Done: {dest} ({os.path.getsize(dest):,} bytes)')

dl('https://legislative.gov.in/sites/default/files/A1860-45.pdf','data/raw/ipc.pdf','IPC PDF')
dl('https://static.pib.gov.in/WriteReadData/userfiles/IndiaTourismStatistics2022English.pdf','data/raw/tourism_stats.pdf','Tourism Stats 2022')

## Step 2 — Download Religious Texts (Gutenberg)

In [ ]:
!wget -q -O data/raw/gita.txt https://www.gutenberg.org/files/54868/54868-0.txt
!wget -q -O data/raw/quran_english.txt https://www.gutenberg.org/files/2800/2800-0.txt
!wget -q -O data/raw/ramayana.txt https://www.gutenberg.org/files/24869/24869-0.txt
print('Religious texts downloaded')

## Step 3 — Wikipedia India Data

In [ ]:
wikipedia.set_lang('en')
pages = ['Kolkata','West Bengal','India','Tourism in India','Victoria Memorial Kolkata',
         'Howrah Bridge','Dakshineswar Temple','Sundarbans','Indian Penal Code',
         'Constitution of India','Hinduism','Islam in India','Sikhism',
         'Archaeological Survey of India','UNESCO World Heritage Sites in India']
texts = []
for t in pages:
    try:
        c = wikipedia.page(t, auto_suggest=False).content
        texts.append(f'# {t}\n\n{c}')
        print(f'  {t}: {len(c):,} chars')
    except Exception as e: print(f'  SKIP {t}: {e}')
    time.sleep(0.3)
with open('data/raw/wiki_india.txt','w',encoding='utf-8') as f: f.write('\n\n---\n\n'.join(texts))
print(f'Saved wiki_india.txt ({len(texts)} pages)')

## Step 4 — Hotels & Tourist Places (Curated Official Data)

In [ ]:
hotels = [
  {'name':'The Oberoi Grand','category':'5-star','city':'Kolkata','price_range':'₹10000-₹25000','lat':22.5574,'lng':88.3506,'source':'Ministry of Tourism'},
  {'name':'ITC Royal Bengal','category':'5-star','city':'Kolkata','price_range':'₹9000-₹20000','lat':22.5175,'lng':88.3637,'source':'Ministry of Tourism'},
  {'name':'Taj Bengal','category':'5-star','city':'Kolkata','price_range':'₹8000-₹18000','lat':22.5373,'lng':88.3334,'source':'Ministry of Tourism'},
  {'name':'Lytton Hotel','category':'3-star','city':'Kolkata','price_range':'₹1800-₹4000','lat':22.5518,'lng':88.3539,'source':'Ministry of Tourism'},
  {'name':'Broadway Hotel','category':'2-star','city':'Kolkata','price_range':'₹800-₹2000','lat':22.5699,'lng':88.3529,'source':'Official listing'},
  {'name':'Hotel Centrum','category':'2-star','city':'Kolkata','price_range':'₹900-₹2200','lat':22.5559,'lng':88.3545,'source':'Official listing'},
]
import json
with open('data/raw/kolkata_hotels.json','w') as f: json.dump(hotels,f,ensure_ascii=False,indent=2)
print(f'Saved {len(hotels)} hotels')

In [ ]:
places = [
  {'name':'Victoria Memorial','city':'Kolkata','category':'Heritage','entry':'₹30','timing':'10AM-5PM Mon closed','lat':22.5448,'lng':88.3426},
  {'name':'Howrah Bridge','city':'Kolkata','category':'Landmark','entry':'Free','timing':'24 hours','lat':22.5851,'lng':88.3468},
  {'name':'Dakshineswar Temple','city':'Kolkata','category':'Temple','entry':'Free','timing':'6AM-12:30PM 3PM-8:30PM','lat':22.6551,'lng':88.3578},
  {'name':'Kalighat Temple','city':'Kolkata','category':'Temple','entry':'Free','timing':'5AM-10PM','lat':22.5199,'lng':88.3432},
  {'name':'Indian Museum','city':'Kolkata','category':'Museum','entry':'₹20','timing':'10AM-5PM Mon closed','lat':22.5579,'lng':88.3518},
  {'name':'Sundarbans National Park','city':'South 24 Parganas','category':'Wildlife','entry':'₹60','timing':'Oct-Mar best season','lat':21.9497,'lng':88.8978},
  {'name':'Taj Mahal','city':'Agra','category':'UNESCO Heritage','entry':'₹50','timing':'Sunrise-Sunset Fri closed','lat':27.1751,'lng':78.0421},
  {'name':'Golden Temple','city':'Amritsar','category':'Religious','entry':'Free','timing':'4AM-11PM','lat':31.6200,'lng':74.8765},
]
with open('data/raw/tourist_places.json','w') as f: json.dump(places,f,ensure_ascii=False,indent=2)
print(f'Saved {len(places)} tourist places')

## Step 5 — Public Social Data (Reddit India, GDELT News)

In [ ]:
headers = {'User-Agent':'SarathiAI/1.0'}
all_posts = []
for sub in ['india','kolkata','IndianFood','indiatourism']:
    try:
        r = requests.get(f'https://www.reddit.com/r/{sub}/hot.json?limit=50',headers=headers,timeout=10)
        posts = r.json()['data']['children']
        for p in posts:
            d = p['data']
            all_posts.append({'sub':sub,'title':d.get('title',''),'text':d.get('selftext','')[:300],'score':d.get('score',0)})
        print(f'  r/{sub}: {len(posts)} posts')
        time.sleep(1)
    except Exception as e: print(f'  r/{sub} failed: {e}')
with open('data/raw/social/reddit_india_public.json','w') as f: json.dump(all_posts,f,ensure_ascii=False,indent=2)
print(f'Saved {len(all_posts)} public Reddit posts')

In [ ]:
# GDELT India news (open public dataset, no auth needed)
try:
    r = requests.get('https://api.gdeltproject.org/api/v2/doc/doc?query=India+Kolkata&mode=artlist&maxrecords=50&format=json',timeout=15)
    articles = r.json().get('articles',[])
    with open('data/raw/social/gdelt_india_news.json','w') as f: json.dump(articles,f,ensure_ascii=False,indent=2)
    print(f'Saved {len(articles)} GDELT India news articles')
except Exception as e: print(f'GDELT failed: {e}')

## ✅ Summary

In [ ]:
for p in Path('data/raw').rglob('*'):
    if p.is_file():
        print(f'{p} — {p.stat().st_size:,} bytes')